# R Master v7a Fix1 · Surface Rebalance（保护四肢）

v7a 的体型方向是有效的，但报告暴露出一个工程问题：变形区域只按高度筛选，手臂/手在相同高度时也可能被误改。

Fix1 从 **v2 原始预览**重新构建，不继承 v7a 的误改结果，并增加严格的 Vertex Group 区域门控：

- 可编辑：LowerBodyStretch / Hip.L / Hip.R / UpperLeg.L / UpperLeg.R / Ass.L / Ass.R
- 强保护：UpperArm / LowerArm / Hand
- 验收硬条件：保护组顶点实际位移必须为 0
- 保持 505 骨，不改骨长，不烘焙 Rest Pose，不导出最终 VRM
- 输出同样的 8 视图：正 / 侧 / 背 / 3/4 + 4 个局部诊断图
- Drive 直读 + 断点续跑


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, json, os

BUILD_TAG="v7a_fix1_groupmask_20260919"

print("R Master v7a Fix1 · 四肢保护版")
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v7a_fix1"/"latest"
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v2 预览文件。截图给二蛋即可。")
print(f"✓ v2 源：{SRC.stat().st_size/1024/1024:.1f} MiB")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v7a_fix1")
LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")



In [ ]:
BUILD=LOCAL/"R_Master_v7a_Fix1_Build.py"
RENDER=LOCAL/"R_Master_v7a_Fix1_Render.py"
BUILD.write_text("\nimport bpy, os, sys, json\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; tag=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--tag\" and i+1<len(argv): tag=argv[i+1]\nif not out: raise RuntimeError(\"missing --out\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or body.type!=\"MESH\": raise RuntimeError(\"R2_Mona_Main missing\")\nif not rig or rig.type!=\"ARMATURE\": raise RuntimeError(\"armature missing\")\n\nfor obj in list(bpy.data.objects):\n    if obj.type==\"MESH\" and obj!=body:\n        bpy.data.objects.remove(obj,do_unlink=True)\n\ndisabled=[]\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"} or \"mask\" in mod.name.lower():\n        disabled.append({\"name\":mod.name,\"type\":mod.type})\n        mod.show_viewport=False; mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1); mod.render_levels=min(mod.render_levels,1)\n\nmesh=body.data\nmw=body.matrix_world.copy(); imw=mw.inverted()\norig=[mw@v.co for v in mesh.vertices]\nmn=Vector((min(p.x for p in orig),min(p.y for p in orig),min(p.z for p in orig)))\nmx=Vector((max(p.x for p in orig),max(p.y for p in orig),max(p.z for p in orig)))\ncenter=(mn+mx)*.5\nheight=max(1e-6,mx.z-mn.z)\ndepth=max(1e-6,mx.y-mn.y)\n\nEDIT_GROUPS=[\"LowerBodyStretch\",\"Hip.L\",\"Hip.R\",\"UpperLeg.L\",\"UpperLeg.R\",\"Ass.L\",\"Ass.R\"]\nPROTECT_GROUPS=[\"UpperArm.L\",\"UpperArm.R\",\"LowerArm.L\",\"LowerArm.R\",\"Hand.L\",\"Hand.R\"]\n\nedit_idx=[body.vertex_groups[g].index for g in EDIT_GROUPS if g in body.vertex_groups]\nprotect_idx=[body.vertex_groups[g].index for g in PROTECT_GROUPS if g in body.vertex_groups]\nif len(edit_idx)<5: raise RuntimeError(\"expected lower-body vertex groups missing\")\nif len(protect_idx)<6: raise RuntimeError(\"expected arm/hand protection groups missing\")\n\ndef weights(v):\n    d={g.group:g.weight for g in v.groups}\n    edit=max([d.get(i,0.0) for i in edit_idx] or [0.0])\n    protect=max([d.get(i,0.0) for i in protect_idx] or [0.0])\n    return edit,protect\n\nposterior_sign=1.0\nass=rig.pose.bones.get(\"Assbutt.L\") or rig.pose.bones.get(\"Ass.L\")\nfront=rig.pose.bones.get(\"Labia.L\") or rig.pose.bones.get(\"LowerBodyStretch\")\nif ass and front:\n    ay=(rig.matrix_world@ass.head).y; fy=(rig.matrix_world@front.head).y\n    posterior_sign=1.0 if ay>=fy else -1.0\n\ndef smoothstep(a,b,x):\n    if a==b: return 1.0 if x>=b else 0.0\n    t=max(0.0,min(1.0,(x-a)/(b-a)))\n    return t*t*(3-2*t)\n\ndef bell(x,a,b,c,d):\n    return smoothstep(a,b,x)*(1-smoothstep(c,d,x))\n\nmoved=0; max_disp=0.0; sum_disp=0.0\nprotected_moved=0; protected_max=0.0\nregion_counts={\"hip\":0,\"waist\":0,\"thigh\":0,\"glute\":0}\n\nfor i,v in enumerate(mesh.vertices):\n    p=orig[i].copy(); q=p.copy()\n    edit_w,protect_w=weights(v)\n    if edit_w<=1e-4 or protect_w>1e-4:\n        continue\n\n    # smooth gate reduces deformation on weakly-weighted boundary vertices\n    gate=smoothstep(0.05,0.55,edit_w)\n    t=(p.z-mn.z)/height\n    xrel=p.x-center.x; yrel=p.y-center.y; ax=abs(xrel)\n\n    hip=bell(t,.405,.445,.535,.575)\n    waist=bell(t,.535,.565,.605,.635)\n    thigh=bell(t,.345,.375,.425,.455)\n    post=posterior_sign*yrel\n    glute=bell(t,.39,.425,.515,.555)*smoothstep(.08*depth,.24*depth,post)\n\n    if hip>0:\n        # modest 8.5% max instead of v7a 9.5%\n        q.x=center.x+xrel*(1-.085*hip*gate)\n        region_counts[\"hip\"]+=1\n    if waist>0:\n        q.x += xrel*(.016*waist*gate)\n        region_counts[\"waist\"]+=1\n    if thigh>0:\n        q.x += xrel*(.014*thigh*gate)\n        region_counts[\"thigh\"]+=1\n    if glute>0:\n        q.y=center.y+yrel*(1-.050*glute*gate)\n        region_counts[\"glute\"]+=1\n\n    delta=q-p; d=delta.length\n    if d>.018:\n        delta*=.018/d; q=p+delta; d=.018\n    if d>1e-6:\n        moved+=1; sum_disp+=d; max_disp=max(max_disp,d)\n        v.co=imw@q\n\n# invariant audit: no protected upper-limb vertex may move relative to v2 source\nnew=[mw@v.co for v in mesh.vertices]\nfor i,v in enumerate(mesh.vertices):\n    _,protect_w=weights(v)\n    if protect_w>1e-4:\n        d=(new[i]-orig[i]).length\n        if d>1e-7:\n            protected_moved+=1; protected_max=max(protected_max,d)\n\nif protected_moved!=0:\n    raise RuntimeError(f\"PROTECTION_FAIL: {protected_moved} protected arm/hand verts moved; max={protected_max}\")\n\nmn2=Vector((min(p.x for p in new),min(p.y for p in new),min(p.z for p in new)))\nmx2=Vector((max(p.x for p in new),max(p.y for p in new),max(p.z for p in new)))\n\nreport={\n \"ok\":True,\n \"stage\":\"R_Master_SurfaceRebalance_v7a_Fix1\",\n \"build_tag\":tag,\n \"source_blend\":bpy.data.filepath,\n \"body_object\":body.name,\n \"rig_object\":rig.name,\n \"bone_count\":len(rig.data.bones),\n \"edit_groups\":EDIT_GROUPS,\n \"protect_groups\":PROTECT_GROUPS,\n \"surface_edit\":{\n   \"moved_vertex_count\":moved,\n   \"total_vertex_count\":len(mesh.vertices),\n   \"mean_displacement_m\":sum_disp/moved if moved else 0.0,\n   \"max_displacement_m\":max_disp,\n   \"safety_cap_m\":.018,\n   \"region_counts\":region_counts\n },\n \"protection_audit\":{\n   \"protected_moved_vertex_count\":protected_moved,\n   \"protected_max_displacement_m\":protected_max,\n   \"pass\":protected_moved==0\n },\n \"bbox_before\":{\"min\":list(map(float,mn)),\"max\":list(map(float,mx))},\n \"bbox_after\":{\"min\":list(map(float,mn2)),\"max\":list(map(float,mx2))},\n \"disabled_modifiers\":disabled,\n \"rest_pose_baked\":False,\n \"final_vrm\":False\n}\nwith open(os.path.join(out,\"R_Master_v7a_Fix1_report.json\"),\"w\",encoding=\"utf-8\") as f:\n    json.dump(report,f,ensure_ascii=False,indent=2)\nbpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,\"R_Master_v7a_Fix1_SURFACE_PREVIEW.blend\"),check_existing=False)\nprint(\"[R Master v7a Fix1] BUILD_OK\")\nprint(\"[R Master v7a Fix1] moved:\",moved)\nprint(\"[R Master v7a Fix1] protected moved:\",protected_moved)\nprint(\"[R Master v7a Fix1] mean/max mm:\",(sum_disp/moved*1000 if moved else 0),max_disp*1000)\n",encoding="utf-8")
RENDER.write_text("\nimport bpy, os, sys\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--view\" and i+1<len(argv): view=argv[i+1]\nif not out or not view: raise RuntimeError(\"missing args\")\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nif not body: raise RuntimeError(\"body missing\")\nfor obj in bpy.context.scene.objects:\n    if obj.type==\"MESH\":\n        obj.hide_render=(obj!=body); obj.hide_viewport=(obj!=body)\n    elif obj.type==\"ARMATURE\": obj.hide_render=True\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"} or \"mask\" in mod.name.lower():\n        mod.show_viewport=False; mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1); mod.render_levels=min(mod.render_levels,1)\npts=[body.matrix_world@Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*.5; height=mx.z-mn.z; dist=max(height,mx.x-mn.x,mx.y-mn.y)*2.5\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"; scene.render.image_settings.file_format=\"PNG\"; scene.render.film_transparent=False\nscene.display.shading.light=\"STUDIO\"; scene.display.shading.show_shadows=True; scene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"; scene.display.shading.color_type=\"SINGLE\"; scene.display.shading.single_color=(.60,.60,.63)\nscene.display.shading.background_type=\"VIEWPORT\"; scene.display.shading.background_color=(.04,.04,.05)\ncd=bpy.data.cameras.get(\"R_v7aF1_Cam_DATA\") or bpy.data.cameras.new(\"R_v7aF1_Cam_DATA\")\ncam=bpy.data.objects.get(\"R_v7aF1_Cam\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_v7aF1_Cam\",cd); scene.collection.objects.link(cam)\nscene.camera=cam; cam.data.type=\"ORTHO\"\ndef look(target): cam.rotation_euler=(Vector(target)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\ndef render(fname,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res; scene.render.resolution_percentage=100\n    cam.location=Vector(pos); cam.data.ortho_scale=scale; look(target)\n    path=os.path.join(out,fname); scene.render.filepath=path; bpy.ops.render.render(write_still=True); return path\nspec={\n \"front\":(\"R_Master_v7a_Fix1_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960)),\n \"side\":(\"R_Master_v7a_Fix1_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960)),\n \"back\":(\"R_Master_v7a_Fix1_back.png\",(center.x,center.y+dist,center.z),center,height*1.08,(720,960)),\n \"three_quarter\":(\"R_Master_v7a_Fix1_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960)),\n}\nfor key,frac in [(\"torso_waist\",.61),(\"waist_pelvis\",.51),(\"pelvis_upperthigh\",.42)]:\n    z=mn.z+height*frac\n    spec[key]=(f\"R_Master_v7a_Fix1_{key}.png\",(center.x,center.y-dist,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\nz=mn.z+height*.49\nspec[\"glute_side\"]=(\"R_Master_v7a_Fix1_glute_side.png\",(center.x+dist,center.y,z),(center.x,center.y,z),max(.52,height*.31),(900,700))\nfname,pos,target,scale,res=spec[view]\nprint(\"[R Master v7a Fix1] RENDER_OK\",view,render(fname,pos,target,scale,res))\n",encoding="utf-8")

STAGE=OUT/"R_Master_v7a_Fix1_SURFACE_PREVIEW.blend"
REPORT=OUT/"R_Master_v7a_Fix1_report.json"
LOG=OUT/"R_Master_v7a_Fix1_build.log"

rebuild=True
if STAGE.exists() and REPORT.exists() and STAGE.stat().st_size>50*1024*1024:
    try:
        rebuild=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except: rebuild=True

if rebuild:
    TMP=LOCAL/"stage"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir(parents=True,exist_ok=True)
    tlog=TMP/"R_Master_v7a_Fix1_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(BUILD),"--","--out",str(TMP),"--tag",BUILD_TAG]
    with tlog.open("w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "R Master v7a Fix1" in line or "Traceback" in line or "Error" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(tlog.read_text(encoding="utf-8",errors="replace")[-10000:])
        raise RuntimeError(f"Fix1 构建失败，退出码 {rc}")
    for n in ["R_Master_v7a_Fix1_SURFACE_PREVIEW.blend","R_Master_v7a_Fix1_report.json","R_Master_v7a_Fix1_build.log"]:
        shutil.copy2(TMP/n,OUT/n)
else:
    print("✓ Fix1 构建已存在，跳过")

r=json.loads(REPORT.read_text(encoding="utf-8"))
print("✓ Protection audit:",r["protection_audit"])
print("✓ moved:",r["surface_edit"]["moved_vertex_count"])



In [ ]:
views=[
 ("front","R_Master_v7a_Fix1_front.png"),
 ("side","R_Master_v7a_Fix1_side.png"),
 ("back","R_Master_v7a_Fix1_back.png"),
 ("three_quarter","R_Master_v7a_Fix1_three_quarter.png"),
 ("torso_waist","R_Master_v7a_Fix1_torso_waist.png"),
 ("waist_pelvis","R_Master_v7a_Fix1_waist_pelvis.png"),
 ("pelvis_upperthigh","R_Master_v7a_Fix1_pelvis_upperthigh.png"),
 ("glute_side","R_Master_v7a_Fix1_glute_side.png"),
]
for i,(view,fname) in enumerate(views,1):
    p=OUT/fname
    if p.exists() and p.stat().st_size>20000:
        print(f"✓ [{i}/8] {view} 已存在，跳过"); continue
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER),"--","--out",str(OUT),"--view",view]
    r=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    important=[x for x in r.stdout.splitlines() if "R Master v7a Fix1" in x or "Traceback" in x or "Error" in x]
    if important: print("\n".join(important[-10:]))
    if r.returncode!=0: raise RuntimeError(f"{view} 渲染失败")
print("✓ Fix1 8 视图完成")



In [ ]:
from IPython.display import display,Image,Markdown
items=[
 ("全身正面","R_Master_v7a_Fix1_front.png"),
 ("全身侧面","R_Master_v7a_Fix1_side.png"),
 ("全身背面","R_Master_v7a_Fix1_back.png"),
 ("全身 3/4","R_Master_v7a_Fix1_three_quarter.png"),
 ("胸廓→腰","R_Master_v7a_Fix1_torso_waist.png"),
 ("腰→骨盆","R_Master_v7a_Fix1_waist_pelvis.png"),
 ("骨盆→大腿根","R_Master_v7a_Fix1_pelvis_upperthigh.png"),
 ("臀线侧视","R_Master_v7a_Fix1_glute_side.png"),
]
for title,f in items:
    display(Markdown(f"### {title}")); display(Image(filename=str(OUT/f),width=500))
zpath=OUT/"R_Master_v7a_Fix1_Review.zip"
if zpath.exists(): zpath.unlink()
with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,f in items: z.write(OUT/f,arcname=f)
    for f in ["R_Master_v7a_Fix1_report.json","R_Master_v7a_Fix1_build.log"]:
        z.write(OUT/f,arcname=f)
print(f"✓ Review ZIP：{zpath.stat().st_size/1024/1024:.1f} MiB")
files.download(str(zpath))

